# ASCENTRA Model Training on Refined Academic Dataset


In [1]:
import numpy as np
import pandas as pd

DATA_PATH = "../data/ascentra_student_data_refined.csv"
TARGET_CANDIDATES = ["academic_risk", "Target", "target", "risk"]

removed_columns = [
    "quiz_average",
    "late_submission_count",
    "lms_resource_access",
    "assignment_average",
    "lms_login_frequency",
    "ct1_score",
    "ct2_score",
]

df = pd.read_csv(DATA_PATH)
target_column = next((col for col in TARGET_CANDIDATES if col in df.columns), df.columns[-1])

print("Dataset shape:", df.shape)
print("Target column:", target_column)
print(df.head())
print("Deleted features remaining:", sorted(set(removed_columns).intersection(df.columns)))


Dataset shape: (4424, 16)
Target column: academic_risk
  student_id  Curricular units 1st sem (approved)  \
0    STU0001                                    0   
1    STU0002                                    6   
2    STU0003                                    0   
3    STU0004                                    6   
4    STU0005                                    5   

   Curricular units 1st sem (grade)  Curricular units 1st sem (enrolled)  \
0                          0.000000                                    0   
1                         14.000000                                    6   
2                          0.000000                                    6   
3                         13.428571                                    6   
4                         12.333333                                    6   

   Curricular units 1st sem (evaluations)  Admission grade  semester  \
0                                       0            127.3         2   
1                        

In [3]:
main_numeric_features = [
    "attendance_percentage",
    "ca1_score",
    "ca2_score",
    "ca3_score",
    "best_2_ca_average",
    "mid_term_score",
    "previous_semester_tgpa",
]

recomputed_best_2 = np.sort(
    df[["ca1_score", "ca2_score", "ca3_score"]].to_numpy(),
    axis=1,
)[:, -2:].mean(axis=1).round(1)

print("Column names:")
print(df.columns.tolist())
print("\nMissing-value counts:")
print(df.isna().sum())
print("\nDescriptive statistics:")
print(df.describe(include="all"))
print("\nCA/mid-term value ranges:")
print(df[["ca1_score", "ca2_score", "ca3_score", "mid_term_score"]].agg(["min", "max"]))
print("\nSemester distribution:")
print(df["semester"].value_counts().sort_index())
print("\nMissing previous TGPA values:", df["previous_semester_tgpa"].isna().sum())
print("Semester 1 rows:", (df["semester"] == 1).sum())
print("Later-semester missing TGPA values:", df.loc[df["semester"] > 1, "previous_semester_tgpa"].isna().sum())
print("\nCorrelation matrix:")
print(df[main_numeric_features].corr().round(3))
print("\nClass distribution:")
print(df[target_column].value_counts().sort_index())
print(df[target_column].value_counts(normalize=True).sort_index().round(3))
print("\nBest-2 CA average correct:", np.allclose(df["best_2_ca_average"], recomputed_best_2))


Column names:
['student_id', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Admission grade', 'semester', 'attendance_percentage', 'ca1_score', 'ca2_score', 'ca3_score', 'best_2_ca_average', 'mid_term_score', 'previous_semester_tgpa', 'academic_trend', 'academic_risk']

Missing-value counts:
student_id                                  0
Curricular units 1st sem (approved)         0
Curricular units 1st sem (grade)            0
Curricular units 1st sem (enrolled)         0
Curricular units 1st sem (evaluations)      0
Admission grade                             0
semester                                    0
attendance_percentage                       0
ca1_score                                   0
ca2_score                                   0
ca3_score                                   0
best_2_ca_average                           0
mid_term_score                              0

In [4]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["student_id", target_column], errors="ignore")
y = df[target_column]

mask = y.notna()
X = X.loc[mask]
y = y.loc[mask].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Training:", X_train.shape)
print("Testing:", X_test.shape)


X shape: (4424, 14)
y shape: (4424,)
Training: (3539, 14)
Testing: (885, 14)


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_columns = X.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_columns = X.select_dtypes(exclude=["object", "category"]).columns.tolist()

try:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", one_hot_encoder),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numerical_columns),
        ("cat", categorical_pipeline, categorical_columns),
    ]
)

print("Categorical:", categorical_columns)
print("Numerical:", numerical_columns)


Categorical: ['academic_trend']
Numerical: ['Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Admission grade', 'semester', 'attendance_percentage', 'ca1_score', 'ca2_score', 'ca3_score', 'best_2_ca_average', 'mid_term_score', 'previous_semester_tgpa']


C:\Users\SAMSUNG\AppData\Local\Temp\ipykernel_7228\3044747570.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X.select_dtypes(include=["object", "category"]).columns.tolist()


In [6]:
from xgboost import XGBClassifier

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

xgb_ascentra = XGBClassifier(
    n_estimators=250,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
)

xgb_ascentra.fit(X_train_processed, y_train)

print("Processed training:", X_train_processed.shape)
print("Processed testing:", X_test_processed.shape)


Processed training: (3539, 16)
Processed testing: (885, 16)


In [7]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

predictions = xgb_ascentra.predict(X_test_processed)
predicted_probabilities = xgb_ascentra.predict_proba(X_test_processed)[:, 1]

print("Confusion matrix:")
print(confusion_matrix(y_test, predictions))
print("\nClassification report:")
print(classification_report(y_test, predictions))
print("ROC AUC:", round(roc_auc_score(y_test, predicted_probabilities), 3))


Confusion matrix:
[[388 124]
 [206 167]]

Classification report:
              precision    recall  f1-score   support

           0       0.65      0.76      0.70       512
           1       0.57      0.45      0.50       373

    accuracy                           0.63       885
   macro avg       0.61      0.60      0.60       885
weighted avg       0.62      0.63      0.62       885

ROC AUC: 0.671


In [8]:
import joblib

joblib.dump(xgb_ascentra, "../models/ascentra_xgboost.pkl")
joblib.dump(preprocessor, "../models/ascentra_preprocessor.pkl")

print("Saved updated model and preprocessor.")


Saved updated model and preprocessor.


In [9]:
import shap

explainer = shap.TreeExplainer(xgb_ascentra)
shap_values = explainer.shap_values(X_test_processed[:200])
feature_names = preprocessor.get_feature_names_out()

shap_summary = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False)

print(shap_summary.head(10))


                          feature  mean_abs_shap
11            num__mid_term_score       0.304334
13  cat__academic_trend_Declining       0.241198
14  cat__academic_trend_Improving       0.116778
6      num__attendance_percentage       0.106829
12    num__previous_semester_tgpa       0.091839
10         num__best_2_ca_average       0.090177
4            num__Admission grade       0.085304
8                  num__ca2_score       0.063485
7                  num__ca1_score       0.059694
9                  num__ca3_score       0.052833
